In [1]:
import os
import math
import random
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm

In [2]:

def get_2d_sincos_pos_embed(embed_dim, grid_size):
    grid_h = torch.arange(grid_size, dtype=torch.float32)
    grid_w = torch.arange(grid_size, dtype=torch.float32)
    grid_y, grid_x = torch.meshgrid(grid_h, grid_w, indexing='ij')

    dim = embed_dim // 2
    omega = torch.arange(dim // 2, dtype=torch.float32)
    omega = 1.0 / (10000 ** (omega / (dim // 2)))

    out_y = torch.einsum('hw,d->hwd', grid_y, omega)
    out_x = torch.einsum('hw,d->hwd', grid_x, omega)

    emb_y = torch.cat([torch.sin(out_y), torch.cos(out_y)], dim=-1)
    emb_x = torch.cat([torch.sin(out_x), torch.cos(out_x)], dim=-1)

    pos_emb = torch.cat([emb_y, emb_x], dim=-1)
    return pos_emb.reshape(grid_size * grid_size, embed_dim)

def modulate(x, shift, scale):
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

class TimestepEmbedder(nn.Module):
    def __init__(self, hidden_size, frequency_embedding_size=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size)
        )
        self.freq_emb_size = frequency_embedding_size

    def forward(self, t):
        half = self.freq_emb_size // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t[:, None].float() * freqs[None]
        embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        return self.mlp(embedding)

class DiTBlock(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.attn = nn.MultiheadAttention(embed_dim=hidden_size, num_heads=num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)

        mlp_hidden = int(hidden_size * 4)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, mlp_hidden),
            nn.GELU(approximate="tanh"),
            nn.Linear(mlp_hidden, hidden_size)
        )

        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 6 * hidden_size, bias=True)
        )
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)

    def forward(self, x, c):
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.adaLN_modulation(c).chunk(6, dim=1)

        attn_input = modulate(self.norm1(x), shift_msa, scale_msa)
        attn_out, _ = self.attn(attn_input, attn_input, attn_input, need_weights=False)
        x = x + gate_msa.unsqueeze(1) * attn_out

        mlp_input = modulate(self.norm2(x), shift_mlp, scale_mlp)
        x = x + gate_mlp.unsqueeze(1) * self.mlp(mlp_input)
        return x

class FinalLayer(nn.Module):
    def __init__(self, hidden_size, patch_size, out_channels):
        super().__init__()
        self.norm_final = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.linear = nn.Linear(hidden_size, patch_size * patch_size * out_channels, bias=True)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 2 * hidden_size, bias=True)
        )
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)
        nn.init.zeros_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x, c):
        shift, scale = self.adaLN_modulation(c).chunk(2, dim=1)
        x = modulate(self.norm_final(x), shift, scale)
        return self.linear(x)

In [3]:

class DiTSmall(nn.Module):
    def __init__(self, input_size=32, patch_size=2, in_channels=4, hidden_size=384, depth=12, num_heads=6, text_dim=3584):
        super().__init__()
        self.in_channels = in_channels
        self.patch_size = patch_size

        self.x_embedder = nn.Conv2d(in_channels, hidden_size, kernel_size=patch_size, stride=patch_size)
        self.t_embedder = TimestepEmbedder(hidden_size)
        self.text_embedder = nn.Sequential(
            nn.Dropout(p=0.1),
            nn.Linear(text_dim, hidden_size),
            nn.SiLU(),
            nn.Dropout(p=0.1),
            nn.Linear(hidden_size, hidden_size)
        )

        num_patches = (input_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, hidden_size), requires_grad=False)
        self.pos_embed.data.copy_(get_2d_sincos_pos_embed(hidden_size, input_size // patch_size).unsqueeze(0))

        self.blocks = nn.ModuleList([DiTBlock(hidden_size, num_heads) for _ in range(depth)])
        self.final_layer = FinalLayer(hidden_size, patch_size, in_channels)

    def unpatchify(self, x):
        B, T, _ = x.shape
        H = W = int(math.sqrt(T))
        p = self.patch_size
        C = self.in_channels
        x = x.reshape(B, H, W, p, p, C)
        x = torch.einsum('nhwpqc->nchpwq', x)
        return x.reshape(B, C, H * p, W * p)

    def forward(self, x, t, text_emb):
        x = self.x_embedder(x)
        x = x.flatten(2).transpose(1, 2)
        x = x + self.pos_embed

        c = self.t_embedder(t) + self.text_embedder(text_emb)

        for block in self.blocks:
            x = block(x, c)

        x = self.final_layer(x, c)
        return self.unpatchify(x)

In [4]:

def load_local_shards(base_dir="./dataset", is_val=False):
    root = Path(base_dir)
    latent_dir = root / "VAE_latents"
    text_dir = root / "gemma4_40_pooled"

    if is_val:
        latent_files = sorted([p for p in latent_dir.rglob("*.pt") if "latents" in str(p).lower() and "val" in str(p).lower()])
        text_files = sorted([p for p in text_dir.rglob("*.pt") if "gemma" in str(p).lower() and "val" in str(p).lower()])
        desc_latent = "Caching Validation VAE Latents"
        desc_text = "Caching Validation Text Embeddings"
    else:
        latent_files = sorted([p for p in latent_dir.rglob("*.pt") if "latents" in str(p).lower() and "val" not in str(p).lower()])
        text_files = sorted([p for p in text_dir.rglob("*.pt") if "gemma" in str(p).lower() and "val" not in str(p).lower()])
        desc_latent = "Caching Training VAE Latents"
        desc_text = "Caching Training Text Embeddings"

    print(f"Found {len(latent_files)} latent shard files.")
    print(f"Found {len(text_files)} Gemma text shard files.")

    latents_dict = {}
    for lf in tqdm(latent_files, desc=desc_latent):
        data = torch.load(lf, map_location="cpu", weights_only=False)
        if isinstance(data, dict):
            latents_dict.update(data)
        elif isinstance(data, list):
            for item in data:
                latents_dict[item[0]] = item[1]

    text_dict = {}
    for tf in tqdm(text_files, desc=desc_text):
        data = torch.load(tf, map_location="cpu", weights_only=False)
        if isinstance(data, dict):
            text_dict.update(data)
        elif isinstance(data, list):
            for item in data:
                text_dict[item[0]] = item[1]

    common_ids = sorted(list(set(latents_dict.keys()) & set(text_dict.keys())))
    print(f"Paired {len(common_ids)} matching samples in RAM.")
    return common_ids, latents_dict, text_dict


class FlowMatchingShardDataset(Dataset):
    def __init__(self, ids, latents_dict, text_dict):
        self.ids = ids
        self.latents_dict = latents_dict
        self.text_dict = text_dict

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        latent = self.latents_dict[img_id]
        text = self.text_dict[img_id]

        if not isinstance(latent, torch.Tensor):
            latent = torch.tensor(latent, dtype=torch.float32)
        else:
            latent = latent.float()

        if latent.ndim == 4 and latent.shape[0] == 1:
            latent = latent.squeeze(0)

        # Dynamic Augmentation
        if random.random() > 0.5:
            latent = torch.flip(latent, dims=[-1])

        # Dynamic Augmentation: Sample 1 of 5 captions randomly
        if isinstance(text, torch.Tensor):
            if text.ndim >= 2:
                chosen_idx = random.randint(0, text.shape[0] - 1)
                chosen_text = text[chosen_idx].float()
            else:
                chosen_text = text.float()
        elif isinstance(text, list):
            chosen_idx = random.randint(0, len(text) - 1)
            chosen_text = torch.tensor(text[chosen_idx], dtype=torch.float32)
        else:
            chosen_text = torch.tensor(text, dtype=torch.float32)

        if chosen_text.ndim == 2:
            chosen_text = chosen_text.mean(dim=0)

        return latent, chosen_text


In [5]:
import os
import copy
import csv
import glob
from torch.optim.lr_scheduler import LambdaLR

def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return LambdaLR(optimizer, lr_lambda)

def run_training(resume_training=True):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Running on local device: {torch.cuda.get_device_name(device) if torch.cuda.is_available() else device}")

    dataset_dir = "/home/incor/Dataset"
    checkpoint_dir = "/home/incor/Dataset/checkpoints"
    os.makedirs(checkpoint_dir, exist_ok=True)
    csv_log_path = os.path.join(checkpoint_dir, "training_log.csv")
    
    print("\n--- Loading Training Data ---")
    common_ids_train, latents_dict_train, text_dict_train = load_local_shards(dataset_dir, is_val=False)
    train_dataset = FlowMatchingShardDataset(common_ids_train, latents_dict_train, text_dict_train)

    print("\n--- Loading Validation Data ---")
    common_ids_val, latents_dict_val, text_dict_val = load_local_shards(dataset_dir, is_val=True)
    val_dataset = FlowMatchingShardDataset(common_ids_val, latents_dict_val, text_dict_val)

    if len(train_dataset) == 0:
        print("Training Dataset is empty. Check your shard paths.")
        return

    sample_latent, sample_text = train_dataset[0]
    inferred_text_dim = sample_text.shape[-1]
    
    train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
    val_dataloader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=0, pin_memory=True, drop_last=False)

    model = DiTSmall(text_dim=inferred_text_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    
    ema_model = copy.deepcopy(model)
    for param in ema_model.parameters():
        param.requires_grad = False
    ema_decay = 0.999

    epochs = 40
    total_steps = len(train_dataloader) * epochs
    warmup_steps = int(0.05 * total_steps)
    
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    start_epoch = 0

    # --- RESUME FROM CHECKPOINT LOGIC ---
    if resume_training:
        checkpoints = glob.glob(os.path.join(checkpoint_dir, "dit_small_epoch_*.pt"))
        if checkpoints:
            latest_checkpoint = max(checkpoints, key=os.path.getctime)
            print(f"\nResuming from checkpoint: {latest_checkpoint}")
            ckpt = torch.load(latest_checkpoint, map_location=device)
            model.load_state_dict(ckpt["model_state_dict"])
            ema_model.load_state_dict(ckpt["ema_model_state_dict"])
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            start_epoch = ckpt["epoch"]
            print(f"Resuming at epoch {start_epoch + 1}")
    
    # Initialize CSV Log
    if start_epoch == 0 or not os.path.exists(csv_log_path):
        with open(csv_log_path, mode="w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["Epoch", "Train_MSE", "Val_MSE", "Learning_Rate"])

    step_count = 0
    for epoch in range(start_epoch, epochs):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{epochs} [Train]")

        for x1, text_emb in pbar:
            x1 = x1.to(device, non_blocking=True)
            text_emb = text_emb.to(device, non_blocking=True)
            B = x1.shape[0]

            x0 = torch.randn_like(x1)
            u = torch.normal(mean=0.0, std=1.0, size=(B,), device=device)
            t = torch.sigmoid(u)
            t_expanded = t.view(B, 1, 1, 1)

            x_t = (1.0 - t_expanded) * x0 + t_expanded * x1
            v_target = x1 - x0

            if torch.rand(1).item() < 0.1:
                text_emb = torch.zeros_like(text_emb)

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                v_pred = model(x_t, t, text_emb)
                loss = F.mse_loss(v_pred, v_target)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                for param, ema_param in zip(model.parameters(), ema_model.parameters()):
                    ema_param.data.mul_(ema_decay).add_(param.data, alpha=1.0 - ema_decay)

            running_loss += loss.item()
            step_count += 1
            current_lr = scheduler.get_last_lr()[0]
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{current_lr:.2e}")

        avg_loss = running_loss / len(train_dataloader)
        
        # --- VALIDATION LOOP ---
        ema_model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x1_val, text_emb_val in tqdm(val_dataloader, desc=f"Epoch {epoch + 1}/{epochs} [Val]"):
                x1_val = x1_val.to(device, non_blocking=True)
                text_emb_val = text_emb_val.to(device, non_blocking=True)
                B = x1_val.shape[0]
                
                x0_val = torch.randn_like(x1_val)
                u = torch.normal(mean=0.0, std=1.0, size=(B,), device=device)
                t_val = torch.sigmoid(u)
                t_expanded = t_val.view(B, 1, 1, 1)
                
                x_t_val = (1.0 - t_expanded) * x0_val + t_expanded * x1_val
                v_target_val = x1_val - x0_val
                
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    v_pred_val = ema_model(x_t_val, t_val, text_emb_val)
                    loss = F.mse_loss(v_pred_val, v_target_val)
                val_loss += loss.item()
                
        avg_val_loss = val_loss / max(1, len(val_dataloader))
        print(f"--- Epoch {epoch + 1} Complete | Train MSE: {avg_loss:.5f} | Val MSE: {avg_val_loss:.5f} ---")

        # --- LOGGING TO CSV ---
        with open(csv_log_path, mode="a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([epoch + 1, avg_loss, avg_val_loss, current_lr])

        # --- SAVE CHECKPOINT ---
        if (epoch + 1) % 5 == 0 or (epoch + 1) == epochs:
            checkpoint_path = os.path.join(checkpoint_dir, f"dit_small_epoch_{epoch + 1}.pt")
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "ema_model_state_dict": ema_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "loss": avg_loss,
                "val_loss": avg_val_loss,
                "text_dim": inferred_text_dim
            }, checkpoint_path)
            print(f"Checkpoint saved: {checkpoint_path}")

    # --- SAVE FINAL MODEL ---
    final_model_dir = os.path.join(checkpoint_dir, "final_model")
    os.makedirs(final_model_dir, exist_ok=True)
    final_path = os.path.join(final_model_dir, "dit_small_final.pt")
    torch.save({
        "ema_model_state_dict": ema_model.state_dict(),
        "text_dim": inferred_text_dim
    }, final_path)
    print(f"\nTraining Complete! Final Inference Model saved to: {final_path}")

run_training(resume_training=True)


Running on local device: NVIDIA GeForce RTX 5060 Ti

--- Loading Training Data ---
Found 12 latent shard files.
Found 12 Gemma text shard files.


Caching Training Text Embeddings: 100%|██████████| 12/12 [00:06<00:00,  1.78it/s]


Paired 118287 matching samples in RAM.

--- Loading Validation Data ---
Found 1 latent shard files.
Found 1 Gemma text shard files.


Caching Validation Text Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.86it/s]


Paired 5000 matching samples in RAM.

Resuming from checkpoint: /home/incor/Dataset/checkpoints/dit_small_epoch_15.pt
Resuming at epoch 16


...

--- Epoch 21 Complete | Train MSE: 0.84610 | Val MSE: 0.84330 ---


Epoch 22/40 [Val]: 100%|██████████| 40/40 [00:12<00:00,  3.12it/s]


--- Epoch 22 Complete | Train MSE: 0.84474 | Val MSE: 0.84119 ---


Epoch 23/40 [Train]:  84%|████████▎ | 773/924 [47:48<09:20,  3.71s/it, loss=0.8242, lr=4.24e-05] 


KeyboardInterrupt: 